# Cost and budget

What a run costs here, how replication competes with exploration, and what stopping early saves.

The live cells are small and take a few minutes on a laptop CPU. Campaign-scale numbers are read from `results/v2/summary.json`. Everything here measures injected noise in one simple simulator setup, so none of it says anything about a real vehicle.

**Matching assessment**

- [Cost and benchmarking](https://niharvaghela1995.github.io/masters-thesis/assessments/x_cost_and_benchmarking.html)

In [1]:
import json, sys, tempfile, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from stp import runner, scenario as S
from stp.v2.config import CANDIDATES, LEVELS, cfg_for
from stp.v2 import rules as RL
SPACE = CANDIDATES["A"]
SUMMARY = json.load(open(ROOT / "results/v2/summary.json"))
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
BLUE, ORANGE = "#2a78d6", "#eb6834"
def along(speeds, k1=0.012, k2=-0.004, gust=0.8):
    return [S.Params(float(v), k1, k2, gust) for v in speeds]
def reruns(p, n, level="medium", base=0):
    return [runner.run(p, base + i, LEVELS[level]) for i in range(n)]


## X1-P01: cost in a form that can be planned with

Time per run, measured live, across the scenario space.

In [2]:
g = np.random.default_rng(1); X = g.uniform(SPACE["xl"], SPACE["xu"], (200, 4)); T, F, St = [], [], []
for i, x in enumerate(X):
    t = time.perf_counter(); r = runner.run(S.Params(*map(float, x)), i, LEVELS["medium"]); T.append(time.perf_counter() - t); F.append(r["failed"]); St.append(r["steps"])
T, F = np.array(T), np.array(F)
print(f"mean {1000 * T.mean():.0f} ms per run, {len(T) / T.sum():.0f} runs/s on one process; passed runs {1000 * T[~F].mean():.0f} ms, failed runs {1000 * T[F].mean():.0f} ms")
print("runs in the v2 campaign:", {k: sum(v for n, v in SUMMARY["runs"].items() if n.startswith(k)) for k in ("gate", "truth", "search")})

mean 68 ms per run, 15 runs/s on one process; passed runs 76 ms, failed runs 38 ms
runs in the v2 campaign: {'gate': 21600, 'truth': 164640, 'search': 187212}


## X1-P02: splitting one budget between exploration and replication

A calculation from saved results. For a total budget of runs, each reported failure costs the rule's mean reruns, so replication reduces what exploration can buy.

In [3]:
B = 2000
for l in ("low", "medium", "high"):
    s = SUMMARY[l]["search"]; Rl = s["scores"]["0.5"]["rules"]
    for arm in ("nsga2", "random"):
        rate = s["reported"][arm] / 4000
        for rule in ("majority-of-19", "SPRT 0.2/0.8, a=b=0.05"):
            c = Rl[rule]["all"]["reruns"]["est"]; e = B / (1 + rate * c)
            print(f"{l:6s} {arm:6s} {rule:24s} of {B} runs: explore {e:6.0f}, replicate {B - e:6.0f}, reported failures {rate * e:6.0f}")

low    nsga2  majority-of-19           of 2000 runs: explore    120, replicate   1880, reported failures     99
low    nsga2  SPRT 0.2/0.8, a=b=0.05   of 2000 runs: explore    562, replicate   1438, reported failures    463
low    random majority-of-19           of 2000 runs: explore    386, replicate   1614, reported failures     85
low    random SPRT 0.2/0.8, a=b=0.05   of 2000 runs: explore   1188, replicate    812, reported failures    261
medium nsga2  majority-of-19           of 2000 runs: explore    119, replicate   1881, reported failures     99
medium nsga2  SPRT 0.2/0.8, a=b=0.05   of 2000 runs: explore    556, replicate   1444, reported failures    463
medium random majority-of-19           of 2000 runs: explore    369, replicate   1631, reported failures     86
medium random SPRT 0.2/0.8, a=b=0.05   of 2000 runs: explore   1159, replicate    841, reported failures    270
high   nsga2  majority-of-19           of 2000 runs: explore    118, replicate   1882, reported failures

## X1-P04: ending a run early

The runner stops once the car is 3 m past the lane edge. The comparison is the steps used against the steps a run to the road's end would take.

In [4]:
X2 = X[F][:60]; used = np.array([St[i] for i in np.flatnonzero(F)[:60]]); full = np.array([int(((S.ROAD_LENGTH - 5) / x[0] + 2) / runner.DT) for x in X2])
print(f"failed runs: mean steps used {used.mean():.0f} of a possible {full.mean():.0f} ({100 * (1 - used.sum() / full.sum()):.0f}% saved)")

failed runs: mean steps used 489 of a possible 871 (44% saved)


## X1-P05: parallel execution

The same batch on one and on two workers. The number is machine-dependent.

In [5]:
from stp.v2.jobs import run_jobs
d = Path(tempfile.mkdtemp()); P = S.Params(15.0, 0.01, -0.01, 0.2).dict()
for w in (1, 2):
    t = time.time(); run_jobs(d, f"w{w}", [("medium", P, i, 0, f"w{w}") for i in range(300)], d / f"w{w}.jsonl", workers=w); print(w, "workers:", round(300 / (time.time() - t), 1), "runs/s")

2026-11-10 13:27:19 w1 total 300 done 0 to run 300


2026-11-10 13:27:40 w1 finished 300 21 s


1 workers: 14.1 runs/s
2026-11-10 13:27:40 w2 total 300 done 0 to run 300


2026-11-10 13:27:51 w2 finished 300 11 s


2 workers: 27.6 runs/s
